In [16]:
import numpy as np
datapath = "../../Preprocessing/Encoded_Data/ecoli_rna_onehot_encoded.npz"
data = np.load(datapath)
embeddings = data["embeddings"]
labels = data["labels"]

In [17]:
from sklearn.cluster import HDBSCAN
from sklearn.metrics import v_measure_score

hdbscan = HDBSCAN()

In [18]:
embeddings.shape

(161, 400)

In [19]:
best_score = 0
best_params = {}
best_results = {}

for min_cluser_size in [5, 10, 20, 30, 50]:
    for min_samples in [1, 5, 10]:
        for metric in ["euclidean", "manhattan", "cosine"]:
            clusterer = HDBSCAN(
                min_cluster_size=min_cluser_size,
                min_samples=min_samples,
                metric=metric,
                copy=True
            )
            hdb_labels = clusterer.fit_predict(embeddings)

            if len(set(hdb_labels)) == 1: # only noise
                continue

            score = v_measure_score(labels, hdb_labels)

            if score > best_score:
                best_score = score
                best_params = {
                    'min_cluster_size': min_cluser_size,
                    'min_samples': min_samples,
                    'metric': metric,
                }
                best_results = {
                    'Number of cluster': len(set(hdb_labels)),
                    'Number of noise points': sum(hdb_labels == -1)
                }

print("Best score:", best_score)
print("Best parameters:", best_params)
print("Best result:", best_results)

Best score: 0.5265566691622177
Best parameters: {'min_cluster_size': 20, 'min_samples': 1, 'metric': 'cosine'}
Best result: {'Number of cluster': 4, 'Number of noise points': np.int64(93)}


### Soft prediction for no -1 label

In [20]:
import hdbscan

best_score = 0
best_params = {}
best_results = {}

for min_cluster_size in [5, 10, 20, 30, 50]:
    for min_samples in [1, 5, 10]:
        for metric in ["euclidean", "manhattan"]:
            clusterer = hdbscan.HDBSCAN(
                    min_cluster_size=min_cluster_size,
                    min_samples=min_samples,
                    metric=metric,
                    prediction_data=True
                )

            predictions = clusterer.fit_predict(embeddings)

            soft_clusters = hdbscan.all_points_membership_vectors(clusterer)
            if soft_clusters.ndim == 1 or soft_clusters.shape[1] <= 1:
                no_noise_predictions = predictions
            else:
                best_cluster = np.argmax(soft_clusters, axis=1)
                no_noise_predictions = np.where(predictions == -1, best_cluster, predictions)

            if len(set(no_noise_predictions)) == 1: # only noise
                continue

            score = v_measure_score(labels, no_noise_predictions)

            if score > best_score:
                best_score = score
                best_params = {
                    'min_cluster_size': min_cluster_size,
                    'min_samples': min_samples,
                    'metric': metric,
                }
                best_results = {
                    'Number of cluster': len(set(no_noise_predictions)),
                    'Number of noise points': sum(no_noise_predictions == -1)
                }

print("Best score:", best_score)
print("Best parameters:", best_params)
print("Best result:", best_results)

Best score: 0.44701812525017076
Best parameters: {'min_cluster_size': 10, 'min_samples': 1, 'metric': 'euclidean'}
Best result: {'Number of cluster': 4, 'Number of noise points': np.int64(0)}
